## 回测结果查看

In [ ]:
import pandas as pd
from chanlun.backtesting import backtest
from chanlun.config import get_data_path
from chanlun.strategy.strategy_3freq_tupo import Strategy3FreqTupo
import os
file_path = "D:/xuangu_log/a_strategy_3freq_tupo/"


In [ ]:
# 从保存的回测落地文件中加载数据
# save_file = str(get_data_path() / "backtest" / "a_w_mmd_v0_signal.pkl")
# save_path = 'D:/xuangu_log/a_strategy_Dnbbc_20250114/'
# save_file = "D:/xuangu_log/a_strategy_Dnbbc_20250107/a_strategy_Dnbbc_20250107.pkl"
save_file = "D:/xuangu_log/a_strategy_3freq_tupo/a_strategy_3freq_tupo.pkl"
file_path = os.path.dirname(save_file)

BT = backtest.BackTest()
BT.load(save_file)
# 显示回测配置与结果
BT.info()
BT.result()

In [ ]:
file_path

In [ ]:
# 保存一个修改持仓附加信息的策略文件
BT.save_file = file_path + '/_add_pos.pkl'
BT.strategy = None
for _code, _poss in BT.trader.positions_history.items():
    for _p in _poss:
        print(_p.loss_price)
        print(_p.price)
        _p.info["loss_rate"] = (_p.price - _p.loss_price) / _p.price * 100
        print(_p.info["loss_rate"])
# print(BT.trader.balance_history)
# print(BT.trader.hold_profit_history)
# print(BT.trader.positions_balance_history)
print(BT.orders)
BT.save()

In [ ]:
# 保存一个不包含策略对象与持仓附加信息的策略文件
BT.save_file = file_path + '_no_strategy.pkl'
BT.strategy = None
for _code, _poss in BT.trader.positions_history.items():
    for _p in _poss:
        _p.info = {}
BT.trader.balance_history = {}
BT.trader.hold_profit_history = {}
BT.trader.positions_balance_history = {}
BT.orders = {}
BT.save()

In [ ]:
# 回测数据展示
BT.backtest_charts()

In [ ]:
# 显示历史持仓
# 设置显示全部行，不省略
pd.set_option("display.max_rows", None)
# 设置显示全部列，不省略
pd.set_option("display.max_columns", None)

# 读取策略中包含的附加信息与平仓uid信息
info_keys = []
close_uids = []
for _, _poss in BT.trader.positions_history.items():
    for _p in _poss:
        # print(_p.info)
        info_keys += list(_p.info.keys())
        close_uids += list([_or["close_uid"] for _or in _p.close_records])
info_keys = list(sorted(list(set(info_keys))))
close_uids = list(sorted(list(set(close_uids))))
print(info_keys)
print(close_uids)

# 显示指定标的的持仓

pos_df = BT.positions(add_columns=info_keys, close_uids=['clear'])
pos_df['_win'] = pos_df['profit_rate'].apply(lambda p: int(p>0))
pos_df.groupby(['mmd']).agg({'profit_rate': {'mean', 'sum', 'count'}, '_win': {'mean', 'sum', 'count'}})
pos_df
pos_df.to_excel(file_path + '/positions.xlsx')

In [ ]:
# 显示历史持仓
#设置显示全部行，不省略
pd.set_option('display.max_rows',None)
#设置显示全部列，不省略
pd.set_option('display.max_columns',None)

# 读取策略中包含的附加信息与平仓uid信息
info_keys = []
close_uids = []
for _, _poss in BT.trader.positions_history.items():
    for _p in _poss:
        # print(_p.info)
        info_keys += list(_p.info.keys())       
        close_uids += list([_or["close_uid"] for _or in _p.close_records])
info_keys = list(sorted(list(set(info_keys))))
close_uids = list(sorted(list(set(close_uids))))
print(info_keys)
print(close_uids)

# 显示指定标的的持仓

pos_df = BT.positions(add_columns=info_keys, close_uids=['clear'])
pos_df['_win'] = pos_df['profit_rate'].apply(lambda p: int(p>0))
pos_df.groupby(['mmd']).agg({'profit_rate': {'mean', 'sum', 'count'}, '_win': {'mean', 'sum', 'count'}})


In [38]:
# 在保存到Excel前移除时区信息
def save_to_excel_without_timezone(dataframe, filename):
    """
    保存DataFrame到Excel，自动处理时区信息
    """
    df_copy = dataframe.copy()
    for col in df_copy.columns:
        if pd.api.types.is_datetime64tz_dtype(df_copy[col]):
            df_copy[col] = df_copy[col].dt.tz_localize(None)
        elif pd.api.types.is_datetime64_any_dtype(df_copy[col]):
            if hasattr(df_copy[col].dt, 'tz') and df_copy[col].dt.tz is not None:
                df_copy[col] = df_copy[col].dt.tz_localize(None)
    
    df_copy.to_excel(filename, index=False)
filename = file_path +'/positions.xlsx'
save_to_excel_without_timezone(pos_df, filename)

In [ ]:
print(pos_df)

In [ ]:
print(len(pos_df))
# 过滤条件
pos_querys = [
    " k_delta <= 3",
    # " xd4_xcld <= 0.2",
    # " xd4_xcld <= 0.5"
]
for _q in pos_querys:
    pos_df = pos_df.query(_q)
print(len(pos_df))

In [37]:
pos_df = BT.positions(add_columns=info_keys, close_uids=['clear',"利润回调30%"])
pos_df['_win'] = pos_df['profit_rate'].apply(lambda p: int(p>0))
pos_df.groupby(['mmd']).agg({'profit_rate': {'mean', 'sum', 'count'}, '_win': {'mean', 'sum', 'count'}})

profit_rate                         _win              
           count       mean         sum count      mean sum
mmd                                                        
1buy          77  10.120467  779.275987    77  0.298701  23

In [ ]:
# fun = lambda x: (x >= 2)
# x_key = 'xd4_xcld'
# pos_df['_x'] = pos_df[x_key].apply(fun)
groupbys = ['opt_mmd',]
print(len(pos_df))
pos_df.groupby(groupbys).agg({'profit_rate': {'mean', 'sum', 'count'}, '_win': {'count', 'sum', 'mean'}})


In [ ]:
# 显示标的周期的图标
BT.show_charts(BT.codes[0], BT.frequencys[0])